In [0]:
# Configuration - update these to your catalog/schema
CATALOG = "main"
SCHEMA = "default"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.sklearn_basic_example"

In [0]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# Generate simple synthetic data
np.random.seed(42)
n = 200
X = pd.DataFrame({
    "feature_a": np.random.randn(n),
    "feature_b": np.random.randn(n),
})
y = (X["feature_a"] + X["feature_b"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train with autolog — registers to UC automatically
mlflow.sklearn.autolog(
    log_input_examples=True,
    registered_model_name=MODEL_NAME,
)

with mlflow.start_run() as run:
    model = LogisticRegression()
    model.fit(X_train, y_train)
    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {model.score(X_test, y_test):.3f}")

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

# Get latest version and set alias (order_by not supported in UC)
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = max(versions, key=lambda v: int(v.version))
client.set_registered_model_alias(MODEL_NAME, "Champion", latest.version)
print(f"Set alias 'Champion' on version {latest.version}")

# Show the captured environment
import yaml
model_uri = f"models:/{MODEL_NAME}/{latest.version}"
deps = mlflow.pyfunc.get_model_dependencies(model_uri)
print(f"\nCaptured requirements ({deps}):")
with open(deps) as f:
    print(f.read())

In [0]:
# Load from UC by alias
loaded_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@Champion")

# Batch inference on new data
new_data = pd.DataFrame({
    "feature_a": np.random.randn(5),
    "feature_b": np.random.randn(5),
})

predictions = loaded_model.predict(new_data)
new_data["prediction"] = predictions
display(new_data)